### Chains with multiple inputs

In [1]:
from dotenv import load_dotenv
import os

# Print the current working directory to verify where Python is looking for the .env file
print("Current working directory:", os.getcwd())

# Load the .env file
load_dotenv()

# Model name
model_name = "gpt-oss-120b"

# Print the API key (first few characters for security)
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print("API key found:", api_key[:4] + "..." + api_key[-4:])
else:
    print("No API key found")

Current working directory: d:\gitlab\da-225o-deep-learning\langchain\2_Chains
API key found: sk-R...-kEw


In [2]:
from langchain.prompts.prompt import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chains.llm import LLMChain

llm = ChatOpenAI(
	model=model_name,
	api_key=api_key,
	base_url="https://model-broker.aviator-model.bp.anthos.otxlab.net/v1",
)

prompt_template = PromptTemplate(input_variables=["input"], template="Tell me a joke about {input}")
chain = LLMChain(llm=llm, prompt=prompt_template)
chain.invoke(input="a parrot")

C:\Users\amondal\AppData\Local\Temp\ipykernel_29796\2874785836.py:12: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt_template)


{'input': 'a parrot',
 'text': 'Why did the parrot bring a ladder to the bar?\n\nBecause it heard the drinks were on the house and wanted to *wing* it! 🦜🍹'}

In [4]:
prompt_template = PromptTemplate(input_variables=["input", "language"], template="Tell me a joke about {input} in {language}")
chain = LLMChain(llm=llm, prompt=prompt_template)
chain.invoke({"input": "a parrot", "language": "Hindi"})

{'input': 'a parrot',
 'language': 'Hindi',
 'text': 'एक तोता डॉक्टर के पास गया और बोला, “डॉक्टर साहब, मुझे हर बात दो‑बार सुनाई देती है!”  \nडॉक्टर ने हँसते हुए कहा, “अरे! तो फिर क्यों नहीं, मैं भी तुम्हें दो‑बार सुनूँगा—‘कॉफ़ी, कॉफ़ी, कॉफ़ी…’” 😄'}

Chains can be more complex and not all sequential chains will be as simple as passing a single string as an argument and getting a single string as output for all steps in the chain

In [5]:
from langchain.chains.sequential import SequentialChain

# This is an LLMChain to write a review given a dish name and the experience.
prompt_review = PromptTemplate.from_template(
    template="You ordered {dish_name} and your experience was {experience}. Write a review: "
)
chain_review = LLMChain(llm=llm, prompt=prompt_review, output_key="review")

# This is an LLMChain to write a follow-up comment given the restaurant review.
prompt_comment = PromptTemplate.from_template(
    template="Given the restaurant review: {review}, write a follow-up comment: "
)
chain_comment = LLMChain(llm=llm, prompt=prompt_comment, output_key="comment")

# This is an LLMChain to summarize a review.
prompt_summary = PromptTemplate.from_template(
    template="Summarise the review in one short sentence: \n\n {comment}"
)
chain_summary = LLMChain(llm=llm, prompt=prompt_summary, output_key="summary")

# This is an LLMChain to translate a summary into German.
prompt_translation = PromptTemplate.from_template(
    template="Translate the summary to Hindi: \n\n {summary}"
)
chain_translation = LLMChain(
    llm=llm, prompt=prompt_translation, output_key="hindi_translation"
)

In [6]:
overall_chain = SequentialChain(
    chains=[chain_review, chain_comment, chain_summary, chain_translation],
    input_variables=["dish_name", "experience"],
    output_variables=["review", "comment", "summary", "hindi_translation"],
)

overall_chain.invoke({"dish_name": "Pizza Salami", "experience": "It was awful!"})

{'dish_name': 'Pizza Salami',
 'experience': 'It was awful!',
 'review': '**Review – Pizza Salami**\n\n**Rating:** ★☆☆☆☆ (1/5)\n\nI ordered a classic Pizza Salami hoping for a quick, satisfying meal, but what arrived was a complete disappointment. Here’s why:\n\n| Aspect | What I Expected | What I Got | Comments |\n|--------|----------------|------------|----------|\n| **Crust** | Thin, crispy with a slight chew | Hard, soggy, and unevenly baked | The crust was burnt in spots and soft as cardboard in the middle. |\n| **Sauce** | Rich tomato base with herbs | Watery, bland sauce that tasted more like tomato juice than pizza sauce | No seasoning, no depth of flavor. |\n| **Salami** | Generous slices of spicy, cured meat | A few dry, rubbery pieces that looked like cheap deli leftovers | The salami lacked any of its usual peppery kick. |\n| **Cheese** | Melted, gooey mozzarella that stretches | Scant, rubbery cheese that barely melted | It was almost as if the pizza was barely topped at a

### Instead of chaining multiple chains together we can also use an LLM to decide which follow up chain is being used

In [7]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.llm import LLMChain
from langchain.prompts import PromptTemplate
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain.chains.router.multi_prompt_prompt import MULTI_PROMPT_ROUTER_TEMPLATE

positive_template = """You are an AI that focuses on the positive side of things. \
Whenever you analyze a text, you look for the positive aspects and highlight them. \
Here is the text:
{input}"""

neutral_template = """You are an AI that has a neutral perspective. You just provide a balanced analysis of the text, \
not favoring any positive or negative aspects. Here is the text:
{input}"""

negative_template = """You are an AI that is designed to find the negative aspects in a text. \
You analyze a text and show the potential downsides. Here is the text:
{input}"""

In [8]:
prompt_infos = [
    {
        "name": "positive",
        "description": "Good for analyzing positive sentiments",
        "prompt_template": positive_template,
    },
    {
        "name": "neutral",
        "description": "Good for analyzing neutral sentiments",
        "prompt_template": neutral_template,
    },
    {
        "name": "negative",
        "description": "Good for analyzing negative sentiments",
        "prompt_template": negative_template,
    },
]

In [9]:
destination_chains = {}
for p_info in prompt_infos:
    # Here we are creating a dictionary of chains, where the key is the name of the chain and the value is the chain itself.
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = PromptTemplate(template=prompt_template, input_variables=["input"])
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain
destination_chains

{'positive': LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='You are an AI that focuses on the positive side of things. Whenever you analyze a text, you look for the positive aspects and highlight them. Here is the text:\n{input}'), llm=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000002690F35A3C0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002690F36A7B0>, root_client=<openai.OpenAI object at 0x000002690F2902F0>, root_async_client=<openai.AsyncOpenAI object at 0x000002690F35A510>, model_name='gpt-oss-120b', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://model-broker.aviator-model.bp.anthos.otxlab.net/v1'), output_parser=StrOutputParser(), llm_kwargs={}),
 'neutral': LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='Yo

In [10]:
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)
print(destinations_str)

positive: Good for analyzing positive sentiments
neutral: Good for analyzing neutral sentiments
negative: Good for analyzing negative sentiments


In [11]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(destinations=destinations_str)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(), # RouterOutputParser is used to parse the output of the router chain into a dictionary
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

chain = MultiPromptChain(
    router_chain=router_chain,
    destination_chains=destination_chains,
    default_chain=destination_chains["neutral"],
    verbose=True,
)

chain.invoke({"input": "I ordered Pizza Salami for 9.99$ and it was not good!"})

C:\Users\amondal\AppData\Local\Temp\ipykernel_29796\731848241.py:10: LangChainDeprecationWarning: Please see migration guide here for recommended implementation: https://python.langchain.com/docs/versions/migrating_chains/multi_prompt_chain/
  chain = MultiPromptChain(




> Entering new MultiPromptChain chain...
negative: {'input': 'I ordered Pizza Salami for 9.99$ and it was not good!'}
> Finished chain.


{'input': 'I ordered Pizza Salami for 9.99$ and it was not good!',
 'text': '**Negative aspects identified in the statement**\n\n| Aspect | Explanation |\n|--------|-------------|\n| **Product quality** | The reviewer explicitly says the pizza “was not good,” indicating poor taste, texture, or overall satisfaction. |\n| **Value for money** | The price is quoted ($9.99). Coupled with the negative quality comment, it suggests the pizza did not meet the expected value for that cost. |\n| **Expectation mismatch** | By ordering a “Pizza Salami,” the consumer likely expected a certain standard (flavor, portion size, topping distribution). The statement implies that those expectations were not met. |\n| **Emotional tone** | The brief, blunt phrasing (“it was not good!”) conveys frustration or disappointment, hinting at a negative customer experience. |\n| **Lack of detail** | No constructive feedback is given (e.g., “undercooked crust,” “insufficient salami”), which may prevent the vendor fro